#Preparacion de entorno

In [ ]:
# Importar librerías necesarias
import pandas as pd
import random
from datetime import datetime, timedelta
import string
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
spark = SparkSession.builder.appName("BasicosNIF").getOrCreate()

In [ ]:
# Montar Google Drive para acceder a los archivos
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
ruta_output = '/content/drive/MyDrive/MisLaboratorios/Output/Básico/Caso_de_uso_3/'

#Lectura de todos los .parquet

In [ ]:
# Definir rutas de los archivos parquet
ruta1 = '/content/drive/MyDrive/Laboratorios/PySpark/Casos de uso/1. Básico/Caso de Uso 3/2.input/planuno/'
ruta2 = '/content/drive/MyDrive/Laboratorios/PySpark/Casos de uso/1. Básico/Caso de Uso 3/2.input/customers_ba/'
ruta3 = '/content/drive/MyDrive/Laboratorios/PySpark/Casos de uso/1. Básico/Caso de Uso 3/2.input/bajas2021/'

# Cargar los archivos parquet en dataframes
df_plauno = spark.read.parquet(ruta1)
df_customers = spark.read.parquet(ruta2)
df_bajas = spark.read.parquet(ruta3)

# Separar por fecha planuno
planuno_2020 = df_plauno[df_plauno['date_alta'] < '2021-01-01']
planuno_2021 = df_plauno[df_plauno['date_alta'] >= '2021-01-01']

In [ ]:
df_plauno.show()


+-----------+------+-------------------------+---------------------+----------------+----------+-------------+------------------------+---------------------+---------------------+--------------------------+----------------------+------------------+------------------+----------+
|customer_id|status|segment_global_group_desc|planuno_quadrant_name|business_area_id|channel_id|customer_type|mobile_digital_crit_type|income_freelance_type|transac_business_type|soc_insur_asgn_fulflt_type|payroll_asgn_crit_type|tax_pymt_crit_type|planuno_segment_id| date_alta|
+-----------+------+-------------------------+---------------------+----------------+----------+-------------+------------------------+---------------------+---------------------+--------------------------+----------------------+------------------+------------------+----------+
|   17884543|     B|            Transaccional|            Prioridad|            BA02|       505|        PYMES|                       1|                    0|      

In [ ]:
df_plauno.select("planuno_quadrant_name").distinct().show()

+---------------------+
|planuno_quadrant_name|
+---------------------+
|            Vinculado|
|        Transaccional|
|         Previnculado|
|               Básico|
+---------------------+



#Base de clientes Basicos

In [ ]:
# Aplicar operaciones de Spark en el DataFrame de Spark
base = (df_plauno
    .filter((col("business_area_id") == "BA03") & (col("planuno_quadrant_name") == "Básico"))
    .select("customer_id").distinct()
)

base.show()

+-----------+
|customer_id|
+-----------+
|   11244603|
|   38210587|
|   77410947|
|   98059940|
|   23278742|
|   48367260|
|   92118017|
|   47774497|
|   55650516|
|   93361018|
|   52730265|
|   80560131|
|   29715140|
|   35503721|
|   48037494|
|   63136681|
|   87541112|
|   52776266|
|   37058653|
|   85695877|
+-----------+
only showing top 20 rows



#Antiguedad del cliente

In [ ]:
df_antiguedad = (base
      .join(df_customers.select("customer_id",
                                col("cust_entry_date").alias("fecha_alta_cliente")),
            "customer_id", "left")
      .withColumn("hoy", lit("2022-01-30"))
      .withColumn("diferencia_dias",
                  datediff(col("hoy").cast("date"),
                           col("fecha_alta_cliente").cast("date")))
      .withColumn(
          "antiguedad",
          when(col("diferencia_dias") < 183, "< 6 meses")
          .when(col("diferencia_dias") < 365, "6-12 meses")
          .when(col("diferencia_dias") < 730, "1-2 años")
          .when(col("diferencia_dias") < 1825, "2-5 años")
          .otherwise("> 5 años")
      ))

df_antiguedad.show()

+-----------+------------------+----------+---------------+----------+
|customer_id|fecha_alta_cliente|       hoy|diferencia_dias|antiguedad|
+-----------+------------------+----------+---------------+----------+
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|
|   11244603|        2021-09-06|2022-01-30|            146| < 6 meses|
|   11244603|        2021-08-17|2022-01-30|            166| < 6 meses|
|   11244603|        2021-01-26|2022-01-30|            369|  1-2 años|
|   11244603|        2021-04-20|2022-01-30|            285|6-12 meses|
|   11244603|        2020-03-10|2022-01-30|            691|  1-2 años|
|   11244603|        2021-09-07|2022-01-30|            145| < 6 meses|
|   11244603|        2020-06-24|2022-01-30|            585|  1-2 años|
|   11244603|        2020-05-04|2022-01-30|            636|  1-2 años|
|   11244603|        2020-09-30|2022-01-30|            487|  1-2 años|
|   11244603|        2021-09-16|2022-01-30|            136| < 6 meses|
|   11

#Refunidos

In [ ]:
w = Window.partitionBy("customer_id").orderBy(col("date_alta").desc())

refunidos_df = (planuno_2020
    .withColumn("row_number", row_number().over(w))
    .filter(col("row_number") == 1)
    .drop("row_number")
    .select("customer_id", "date_alta"))

refunidos_df.show()

+-----------+----------+
|customer_id| date_alta|
+-----------+----------+
|   10086634|2020-12-26|
|   10135158|2020-10-08|
|   10140584|2020-12-10|
|   10181500|2020-12-14|
|   10236286|2020-08-28|
|   10238600|2020-12-30|
|   10257966|2020-12-16|
|   10340368|2020-10-28|
|   10377933|2020-12-20|
|   10378759|2020-12-22|
|   10385997|2020-12-28|
|   10386001|2020-12-26|
|   10423268|2020-12-26|
|   10430393|2020-11-26|
|   10430500|2020-12-12|
|   10431706|2020-11-17|
|   10443788|2020-12-28|
|   10457346|2020-12-25|
|   10463462|2020-12-08|
|   10467919|2020-12-29|
+-----------+----------+
only showing top 20 rows



#Clasificacion Plan Uno 2021 + flag 3d3

In [ ]:
planuno_2021_enriq = (planuno_2020
    .withColumn(
        "Exclusivos_3d3",
        when(
            (
              (col("customer_type").isin(
                  "B.PRIVADA", "B.PERSONAL", "PARTICULARES", "PAES")) &
              (col("mobile_digital_crit_type") == 1) &
              (col("income_freelance_type") == 0) &
              (col("transac_business_type") == 0)
            ) |
            (
              (col("customer_type") == "PYMES") &
              (col("mobile_digital_crit_type") == 1) &
              (expr("soc_insur_asgn_fulflt_type  + payroll_asgn_crit_type + tax_pymt_crit_type") == 1) &
              (col("planuno_segment_id").isin("PT","AT","CT","BT","VT"))
            ),
            lit("Exclusivos_3d3")
        ).otherwise("Otros")
    )
    .select("customer_id", "planuno_quadrant_name", "Exclusivos_3d3"))

planuno_2021_enriq.show()

+-----------+---------------------+--------------+
|customer_id|planuno_quadrant_name|Exclusivos_3d3|
+-----------+---------------------+--------------+
|   17884543|            Vinculado|Exclusivos_3d3|
|   60590776|        Transaccional|         Otros|
|   94871591|               Básico|Exclusivos_3d3|
|   77635896|         Previnculado|         Otros|
|   95877077|         Previnculado|Exclusivos_3d3|
|   46674711|               Básico|         Otros|
|   41207192|            Vinculado|         Otros|
|   19627628|               Básico|Exclusivos_3d3|
|   81369496|            Vinculado|         Otros|
|   75092544|            Vinculado|         Otros|
|   18025908|               Básico|Exclusivos_3d3|
|   38733114|            Vinculado|Exclusivos_3d3|
|   96199938|               Básico|         Otros|
|   10430393|         Previnculado|Exclusivos_3d3|
|   61640281|        Transaccional|Exclusivos_3d3|
|   53275341|         Previnculado|Exclusivos_3d3|
|   81818881|         Previncul

In [ ]:
planuno_2020.show(2,0)

+-----------+------+-------------------------+---------------------+----------------+----------+-------------+------------------------+---------------------+---------------------+--------------------------+----------------------+------------------+------------------+----------+
|customer_id|status|segment_global_group_desc|planuno_quadrant_name|business_area_id|channel_id|customer_type|mobile_digital_crit_type|income_freelance_type|transac_business_type|soc_insur_asgn_fulflt_type|payroll_asgn_crit_type|tax_pymt_crit_type|planuno_segment_id|date_alta |
+-----------+------+-------------------------+---------------------+----------------+----------+-------------+------------------------+---------------------+---------------------+--------------------------+----------------------+------------------+------------------+----------+
|17884543   |B     |Transaccional            |Prioridad            |BA02            |505       |PYMES        |1                       |0                    |1     

#Union con clasificacion 2021 y bajas

In [ ]:
df = (df_antiguedad.join(planuno_2021_enriq, "customer_id", "left")
      .join(df_bajas.select("customer_id", col("movement_type")), "customer_id", "left")
      .withColumn("bajas", when(col("movement_type") == "baja", 1).otherwise(0))
     )

df.show()

+-----------+------------------+----------+---------------+----------+---------------------+--------------+-------------+-----+
|customer_id|fecha_alta_cliente|       hoy|diferencia_dias|antiguedad|planuno_quadrant_name|Exclusivos_3d3|movement_type|bajas|
+-----------+------------------+----------+---------------+----------+---------------------+--------------+-------------+-----+
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|         Previnculado|         Otros|         baja|    1|
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|            Vinculado|         Otros|         baja|    1|
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|         Previnculado|         Otros|         baja|    1|
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|            Vinculado|         Otros|         baja|    1|
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|            Vinculado|Exclusivos_3

#Columna cuadrante_planuno

In [ ]:
df_cuadrante_planuno = (df
                        .withColumn(
                            "cuadrante_planuno",
                            when(col("bajas") == 1, "BAJAS")
                            .when(
                                (col("planuno_quadrant_name") == "Transaccional") &
                                (col("Exclusivos_3d3") == "Exclusivos_3d3"),
                                "3d3"
                            )
                            .otherwise(col("planuno_quadrant_name"))
                        )
                       )

df_cuadrante_planuno.show()

+-----------+------------------+----------+---------------+----------+---------------------+--------------+-------------+-----+-----------------+
|customer_id|fecha_alta_cliente|       hoy|diferencia_dias|antiguedad|planuno_quadrant_name|Exclusivos_3d3|movement_type|bajas|cuadrante_planuno|
+-----------+------------------+----------+---------------+----------+---------------------+--------------+-------------+-----+-----------------+
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|         Previnculado|         Otros|         baja|    1|            BAJAS|
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|            Vinculado|         Otros|         baja|    1|            BAJAS|
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|         Previnculado|         Otros|         baja|    1|            BAJAS|
|   11244603|        2021-05-10|2022-01-30|            265|6-12 meses|            Vinculado|         Otros|         baja|   

#Resultado

In [ ]:
df_cuadrante_planuno.select("customer_id", "antiguedad", "cuadrante_planuno", "Exclusivos_3d3", "bajas").show()

+-----------+----------+-----------------+--------------+-----+
|customer_id|antiguedad|cuadrante_planuno|Exclusivos_3d3|bajas|
+-----------+----------+-----------------+--------------+-----+
|   11244603|6-12 meses|            BAJAS|         Otros|    1|
|   11244603|6-12 meses|            BAJAS|         Otros|    1|
|   11244603|6-12 meses|            BAJAS|         Otros|    1|
|   11244603|6-12 meses|            BAJAS|         Otros|    1|
|   11244603|6-12 meses|            BAJAS|Exclusivos_3d3|    1|
|   11244603|6-12 meses|            BAJAS|Exclusivos_3d3|    1|
|   11244603|6-12 meses|            BAJAS|Exclusivos_3d3|    1|
|   11244603|6-12 meses|            BAJAS|         Otros|    1|
|   11244603|6-12 meses|            BAJAS|Exclusivos_3d3|    1|
|   11244603|6-12 meses|            BAJAS|Exclusivos_3d3|    1|
|   11244603| < 6 meses|            BAJAS|         Otros|    1|
|   11244603| < 6 meses|            BAJAS|         Otros|    1|
|   11244603| < 6 meses|            BAJA

In [ ]:
df_cuadrante_planuno.groupBy("antiguedad", "cuadrante_planuno").count().orderBy("antiguedad", "cuadrante_planuno").show()

+----------+-----------------+------+
|antiguedad|cuadrante_planuno| count|
+----------+-----------------+------+
|  1-2 años|             NULL|    20|
|  1-2 años|              3d3|100312|
|  1-2 años|            BAJAS|156637|
|  1-2 años|           Básico|182824|
|  1-2 años|     Previnculado|173740|
|  1-2 años|    Transaccional| 75437|
|  1-2 años|        Vinculado|171281|
|  2-5 años|             NULL|     2|
|  2-5 años|              3d3|  9007|
|  2-5 años|            BAJAS| 12819|
|  2-5 años|           Básico| 16461|
|  2-5 años|     Previnculado| 15518|
|  2-5 años|    Transaccional|  6659|
|  2-5 años|        Vinculado| 15212|
|6-12 meses|             NULL|     4|
|6-12 meses|              3d3| 50215|
|6-12 meses|            BAJAS| 79906|
|6-12 meses|           Básico| 91342|
|6-12 meses|     Previnculado| 87559|
|6-12 meses|    Transaccional| 37054|
+----------+-----------------+------+
only showing top 20 rows



#Exportar csv a output

In [ ]:
# Definir la ruta de salida para el archivo csv
csv_path_pyspark = ruta_output + 'cuadrante_planuno/'
csv_path_panda = ruta_output + 'cuadrante_planuno.csv'

# Escribir el dataframe con los KPIs en un directorio csv, sobrescribiendo si existe
df_cuadrante_planuno.coalesce(1).write.csv(csv_path_pyspark, mode="overwrite")

In [ ]:
# Escribir como un fichero csv directamente
df_cuadrante_planuno.toPandas().to_csv(csv_path_panda, index=False)